In [ ]:
import os
import yaml
import polars as pl
import pandas as pd
import numpy as np
from tqdm import tqdm
from plotnine import *

from scripts import get_correlations

In [ ]:
config_path = "/home/dnanexus/ukbgym/config_wgs_absplice2.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)

cov_list = config.get('covariates')

all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

all_annotation_list = list(set(all_annotation_list))
len(all_annotation_list)


In [ ]:
import statsmodels.api as sm

# Get covariate corrected phenotypes
def cov_prs_correction(all_df, phenotypes, covariates, prs_pheno_map):
    # Initialize an empty DataFrame to store residuals
    # all_df.set_index('sample', inplace=True)
    cov_prs_corrected_phenos = pd.DataFrame(
        index=all_df.index
    )  # Index is the sample ID

    # Perform linear regression for each phenotype
    for pheno in tqdm(phenotypes):
        # Drop NaN values for the current phenotype
        combined_df = all_df[[pheno] + covariates + [prs_pheno_map[pheno]]].dropna()
        y = combined_df[pheno]
        X = combined_df.drop(columns=[pheno])
        X = sm.add_constant(X)  # Add a constant term for the intercept

        # Fit the model
        model = sm.OLS(y, X).fit()

        # Save residuals
        residuals = pd.Series(model.resid, index=combined_df.index, name=pheno)
        cov_prs_corrected_phenos = pd.concat(
            [cov_prs_corrected_phenos, residuals], axis=1
        )

    # Reset the index for the resulting DataFrame
    cov_prs_corrected_phenos.reset_index(inplace=True)
    # cov_prs_corrected_phenos.columns = ['sample'] + [f"{pheno}_cov_prs_corrected" for pheno in phenotypes]
    return cov_prs_corrected_phenos

# Get covariate corrected phenotypes
def cov_correction(all_df, phenotypes, covariates):
    # Initialize an empty DataFrame to store residuals
    cov_prs_corrected_phenos = pd.DataFrame(index=all_df.index)  # Index is the sample ID

    # Perform linear regression for each phenotype
    for pheno in tqdm(phenotypes):
        # Drop NaN values for the current phenotype
        combined_df = all_df[[pheno] + covariates].dropna()
        y = combined_df[pheno]
        X = combined_df.drop(columns=[pheno])
        X = sm.add_constant(X)  # Add a constant term for the intercept

        # Fit the model
        model = sm.OLS(y, X).fit()

        # Save residuals
        residuals = pd.Series(model.resid, index=combined_df.index, name=pheno)
        cov_prs_corrected_phenos = pd.concat(
            [cov_prs_corrected_phenos, residuals], axis=1
        )

    # Reset the index for the resulting DataFrame
    cov_prs_corrected_phenos.reset_index(inplace=True)
    # cov_prs_corrected_phenos.columns = ['sample'] + [f"{pheno}_cov_prs_corrected" for pheno in phenotypes]
    return cov_prs_corrected_phenos
    

In [ ]:
config_path = f'/home/dnanexus/ukbgym/config_wgs_absplice2.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

covs = config.get("covariates")

data_dir = '/home/dnanexus/data_dir'
associations_df_path = f'{data_dir}/absplice2_assocs.parquet'
pheno_df_path = f'{data_dir}/250629_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet'

assocs = pl.read_parquet(associations_df_path)
phenotypes = list(assocs['phenotype'].unique())
pheno_df = pl.read_parquet(pheno_df_path, columns=["eid"] + covs + phenotypes).rename({'eid': 'sample_id'})

prs_file = f'{data_dir}/PRS.parquet'
prs_pheno_map_file = f'{data_dir}/prs_pheno_map_clean.csv'
prs_pheno_map = pd.read_csv(prs_pheno_map_file)
prs_pheno_map = dict(zip(prs_pheno_map["phenotype"], prs_pheno_map["pgs_id"]))
prs_df = pl.read_parquet(prs_file).rename({'sample': 'sample_id'})

all_df = pheno_df.join(prs_df, on='sample_id', how='inner').to_pandas()

pheno_corrected_df = pl.from_pandas(cov_prs_correction(all_df.set_index('sample_id'), phenotypes, covs, prs_pheno_map))
pheno_corrected_df

In [ ]:
# Function to compute correlations
def compute_correlations_lazy(
    df_lazy: pl.LazyFrame,
    filter_nan: bool = True,
) -> pl.LazyFrame:
    """
    Computes both Pearson and Spearman correlations:
    - Between ('sum', 'value'), ('max', 'value'), ('top2', 'value')
    - Grouped by ('annotation', 'phenotype')
    """
    # Filter out NaN & null
    if filter_nan:
        df_clean = df_lazy.filter(
            pl.all_horizontal(
                pl.col(["sum", "max", "top2", "value"]).is_not_nan() & 
                pl.col(["sum", "max", "top2", "value"]).is_not_null() &
                pl.col(["sum", "max", "top2", "value"]).is_finite()
            )
        )
    else:
        df_clean = df_lazy.with_columns([
            pl.col("sum").fill_null(0).fill_nan(0),
            pl.col("max").fill_null(0).fill_nan(0),
            pl.col("top2").fill_null(0).fill_nan(0),
            pl.col("value").fill_null(0).fill_nan(0),
        ])

    # Add ranks per group for Spearman
    df_ranks = df_clean.with_columns([
        pl.col("sum").rank().over(["annotation", "phenotype"]).alias("rank_sum"),
        pl.col("max").rank().over(["annotation", "phenotype"]).alias("rank_max"),
        pl.col("top2").rank().over(["annotation", "phenotype"]).alias("rank_top2"),
        pl.col("value").rank().over(["annotation", "phenotype"]).alias("rank_value"),
    ])

    # Group by + compute both sets of correlations
    correlations = df_ranks.group_by(["annotation", "phenotype"]).agg([
        # Pearson
        pl.corr("sum", "value").alias("sum_pearson"),
        pl.corr("max", "value").alias("max_pearson"),
        pl.corr("top2", "value").alias("top2_pearson"),
        # Spearman
        pl.corr("rank_sum", "rank_value").alias("sum_spearman"),
        pl.corr("rank_max", "rank_value").alias("max_spearman"),
        pl.corr("rank_top2", "rank_value").alias("top2_spearman"),
    ])

    return correlations

def compute_gene_correlations(
    data_dir: str,
    burdens_dir: str,
    pheno_corrected_df: pl.DataFrame,
    config: dict,
    filter_nan: bool = True,
    subset_annos: list | None = None,
    subset_samples: list | None = None,
) -> pl.DataFrame:
    """
    Compute correlations between gene burden files and phenotypes,
    add annotation categories, fill NaNs, and return long-format correlations
    """

    # Subset annotations
    subset_annos = []
    rare_variant_annotations_dict = config.get('rare_variant_annotations')
    rare_variant_annotations_dict = {
        k: v for k, v in rare_variant_annotations_dict.items() if k != 'misc'
    } if rare_variant_annotations_dict else {}

    for category in rare_variant_annotations_dict.values():
        subset_annos.extend(category)
    subset_annos = list(set(subset_annos))

    # Build annotation -> category map
    annotation_category_map = {}
    for category, annotations in rare_variant_annotations_dict.items():
        for ann in annotations:
            annotation_category_map[ann] = category

    # Read associations
    assocs = pl.read_parquet(f'{data_dir}/absplice2_assocs.parquet')

    # Prepare phenotype DataFrame
    pheno_df = pheno_corrected_df.unpivot(
        index=['sample_id'],
        variable_name='phenotype',
        value_name='value'
    )

    corr_df_list = []

    for gene_file in tqdm(os.listdir(burdens_dir), desc="Correlations for genes"):
        if not gene_file.endswith('.parquet'):
            continue

        gene_id = gene_file.split('.')[0]
        adf = assocs.filter(pl.col('gene_id') == gene_id)
        phenos_needed = adf['phenotype'].unique().to_list()

        pheno_filtered = (
            pheno_df
            .filter(pl.col('phenotype').is_in(phenos_needed))
            .lazy()
        )

        bdf = pl.scan_parquet(f'{burdens_dir}/{gene_file}')

        if subset_annos is not None:
            bdf = bdf.filter(pl.col('annotation').is_in(subset_annos))

        if subset_samples is not None:
            bdf = bdf.filter(pl.col('sample_id').is_in(subset_samples))

        cdf = bdf.join(pheno_filtered, on='sample_id', how='inner')

        corr_df_list.append(
            compute_correlations_lazy(cdf, filter_nan=filter_nan).with_columns(
                pl.lit(gene_id).alias('gene_id'),
            ).collect()
        )

    corr_df = pl.concat(corr_df_list)

    # Add category column
    corr_df = corr_df.with_columns(
        pl.col("annotation").replace(annotation_category_map).alias("category")
    )

    # Fill NaNs in correlation columns
    corr_cols = [col for col in corr_df.columns if "pearson" in col or "spearman" in col]
    corr_df = corr_df.with_columns(
        [pl.col(col).fill_null(0).alias(col) for col in corr_cols]
    )

    return corr_df

    # Melt to long format
    # corr_long = corr_df.unpivot(
    #     index=["annotation", "phenotype", "gene_id", "category"],
    #     variable_name="correlation_type",
    #     value_name="correlation"
    # ).with_columns([
    #     pl.col("correlation_type").str.extract(r"(pearson|spearman)").alias("method"),
    #     pl.col("correlation_type").str.extract(r"(sum|max|top2)").alias("aggregation"),
    #     pl.col("correlation").abs().alias("abs_correlation")
    # ])

    # return corr_long


In [ ]:
corr_long = compute_gene_correlations(
    data_dir='/home/dnanexus/data_dir',
    burdens_dir='/home/dnanexus/absplice2_61genes/',
    pheno_corrected_df=pheno_corrected_df,
    config=config,
    filter_nan=False,
    subset_annos=['loftee_hc', 'am_pathogenicity', 'pangolin_score', 'AbSplice2_max'],
    subset_samples=None
)

corr_long

In [ ]:
aggregation = "sum"
method = "spearman"


agg_df = (
    corr_long
    .filter(pl.col("aggregation") == aggregation)
    .filter(pl.col("method") == method)
    .group_by("annotation")
    .agg(pl.median("abs_correlation").alias("median_abs_correlation"))
    .sort("median_abs_correlation", descending=True)
)

ordered_annotations = agg_df['annotation'].to_list()

corr_long_pd = corr_long.to_pandas()
# corr_long_pd = corr_long.filter(pl.col('annotation').is_in(subset_annos)).to_pandas()
corr_long_pd['annotation'] = pd.Categorical(
    corr_long_pd['annotation'],
    categories=ordered_annotations,
    ordered=True
)

(
    ggplot(
        corr_long_pd.query(f"aggregation == '{aggregation}' & method == '{method}'"),
        aes(x='annotation', y='abs_correlation', fill='category')
    )
    + geom_boxplot(alpha=0.75)
    + theme_bw()
    # + scale_y_sqrt()
    + ylab('|rank correlation|')
    + theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(12, 8),
    )
)

## Debugging

In [ ]:
data_dir = '/home/dnanexus/data_dir'
# burdens_dir = f"{data_dir}/burdens/absplice2_77_genes/"
burdens_dir = f"/home/dnanexus/absplice2_61genes/"
# subset_samples = pl.read_parquet(f'{data_dir}/167k_sample_ids.parquet')
subset_samples = None

# Subset to a smaller set of annotations
subset_annos = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
rare_variant_annotations_dict = {k: v for k, v in rare_variant_annotations_dict.items() if k != 'misc'}
for category in rare_variant_annotations_dict.values():
    subset_annos.extend(category)
subset_annos = list(set(subset_annos))

assocs = pl.read_parquet(f'{data_dir}/absplice2_assocs.parquet')

# Read once, up front
pheno_df = pheno_corrected_df.unpivot(index=['sample_id'],
             variable_name='phenotype',
             value_name='value')

corr_df_list = []
# In loop: filter using an eager list, then switch to lazy for join
for gene_file in tqdm(os.listdir(burdens_dir), desc="Correlations for genes"):
    if not gene_file.endswith('.parquet'):
        continue

    gene_id = gene_file.split('.')[0]
    adf = assocs.filter(pl.col('gene_id') == gene_id)

    phenos_needed = adf['phenotype'].unique().to_list()

    pheno_filtered = (
        pheno_df
        .filter(pl.col('phenotype').is_in(phenos_needed))
        .lazy()
    )

    bdf = pl.scan_parquet(f'{burdens_dir}/{gene_file}')#.filter(pl.col('annotation').is_in(subset_annos))
    
    # Compute correlations on a subset of sampels
    if subset_samples is not None:
        bdf = bdf.filter(pl.col('sample_id').is_in(subset_samples['sample_id']))

    cdf = bdf.join(pheno_filtered, on='sample_id', how='inner')
    
    corr_df_list.append(
        compute_correlations_lazy(cdf).with_columns(
            pl.lit(gene_id).alias('gene_id')
        ).collect()
    )

corr_df = pl.concat(corr_df_list)
corr_df

In [ ]:
annotation_category_map = {}
if rare_variant_annotations_dict:
    for category, annotations in rare_variant_annotations_dict.items():
        for ann in annotations:
            annotation_category_map[ann] = category

# Add category
corr_df = corr_df.with_columns(
    pl.col("annotation").replace(annotation_category_map).alias("category")
)

# Fill NaNs
corr_cols = [col for col in corr_df.columns if "pearson" in col or "spearman" in col]
corr_df = corr_df.with_columns(
    [pl.col(col).fill_null(0).alias(col) for col in corr_cols]
)

# Melt
corr_long = corr_df.unpivot(
    index=["annotation", "phenotype", "gene_id", "category"],
    variable_name="correlation_type",
    value_name="correlation"
).with_columns([
    pl.col("correlation_type").str.extract(r"(pearson|spearman)").alias("method"),
    pl.col("correlation_type").str.extract(r"(sum|max|top2)").alias("aggregation"),
    pl.col("correlation").abs().alias("abs_correlation")
])
corr_long

In [ ]:
aggregation = "max"
method = "spearman"


agg_df = (
    corr_long
    .filter(pl.col("aggregation") == aggregation)
    .filter(pl.col("method") == method)
    .group_by("annotation")
    .agg(pl.median("abs_correlation").alias("median_abs_correlation"))
    .sort("median_abs_correlation", descending=True)
)

ordered_annotations = agg_df['annotation'].to_list()

corr_long_pd = corr_long.to_pandas()
# corr_long_pd = corr_long.filter(pl.col('annotation').is_in(subset_annos)).to_pandas()
corr_long_pd['annotation'] = pd.Categorical(
    corr_long_pd['annotation'],
    categories=ordered_annotations,
    ordered=True
)

(
    ggplot(
        corr_long_pd.query(f"aggregation == '{aggregation}' & method == '{method}'"),
        aes(x='annotation', y='abs_correlation', fill='category')
    )
    + geom_boxplot(alpha=0.75)
    + theme_bw()
    # + scale_y_sqrt()
    + ylab('|rank correlation|')
    + theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(12, 8),
    )
)